In [3]:
import pandas as pd

# Set the season to analyse
# Replace 'XXX' with the season_id you want to target, e.g. 'f010783f-7fb5-40d8-9e00-6d8a45fa448e'
target_season_id = 'f010783f-7fb5-40d8-9e00-6d8a45fa448e'

# Load data
profiles = pd.read_csv('../CSV/24.09.2026/profiles_rows.csv')
game_week_scores = pd.read_csv('../CSV/24.09.2026/game_week_scores_rows.csv')
game_weeks = pd.read_csv('../CSV/24.09.2026/game_weeks_rows.csv')
season_players = pd.read_csv('../CSV/24.09.2026/season_players_rows.csv')

if target_season_id == 'XXX':
    raise ValueError(
        "Set target_season_id to the season_id you want to analyse before running the notebook."
    )

# Restrict to the selected season and only players in that season
active_player_ids = season_players.loc[
    season_players['season_id'] == target_season_id,
    'player_id'
].unique()

target_game_week_ids = game_weeks.loc[
    game_weeks['season_id'] == target_season_id,
    'id'
].unique()

game_week_scores = game_week_scores[
    game_week_scores['player_id'].isin(active_player_ids)
    & game_week_scores['game_week_id'].isin(target_game_week_ids)
].copy()

profiles = profiles[['id', 'username']].copy()

print(f"Target season: {target_season_id}")
print("Profiles shape:", profiles.shape)
print("Game week scores shape:", game_week_scores.shape)
print("Game weeks shape:", game_weeks.shape)

Target season: f010783f-7fb5-40d8-9e00-6d8a45fa448e
Profiles shape: (66, 2)
Game week scores shape: (220, 6)
Game weeks shape: (43, 8)


In [4]:
# Merge game_week_scores with profiles on player_id -> id
merged = game_week_scores.merge(
    profiles,
    left_on='player_id',
    right_on='id',
    how='inner',
    suffixes=('', '_profile')
)

# Merge with game_weeks on game_week_id -> id to get week_number
merged = merged.merge(
    game_weeks[['id', 'week_number']],
    left_on='game_week_id',
    right_on='id',
    how='left',
    suffixes=('', '_gw')
)

# Remove missing usernames and legacy noise before scoring
after = merged.dropna(subset=['username']).copy()
merged = after[after['username'] != 'Martinez'].copy()

print("Merged data shape:", merged.shape)
print("\nColumns:", merged.columns.tolist())

Merged data shape: (220, 10)

Columns: ['id', 'game_week_id', 'player_id', 'correct_scores', 'points', 'created_at', 'id_profile', 'username', 'id_gw', 'week_number']


In [5]:
# Sort by Points (descending) then Correct Scores (descending)
sorted_scores = merged.sort_values(by=['points', 'correct_scores'], ascending=[False, False])

# Get the maximum points value
max_points = sorted_scores['points'].max()

# Filter to only rows with the maximum points (top scoring game week(s))
top_scoring = sorted_scores[sorted_scores['points'] == max_points].copy()

# Select relevant columns for display
display_columns = ['username', 'points', 'correct_scores', 'week_number']
top_scoring_display = top_scoring[display_columns].reset_index(drop=True)

# Rename columns for better display
top_scoring_display.columns = ['Player Name', 'Points', 'Correct Scores', 'Game Week']

print(f"Top Scoring Game Week(s) - Points: {max_points}")
print(top_scoring_display)

Top Scoring Game Week(s) - Points: 13
  Player Name  Points  Correct Scores  Game Week
0     Murphty      13               3          4
1     Murphty      13               3          3


In [6]:
# Create a styled display like the previous notebooks
styled = top_scoring_display.style.set_table_styles([
    {'selector': 'thead th', 'props': [('background-color', '#4472C4'), ('color', 'white'), ('padding', '8px'), ('border', '1px solid black')]},
    {'selector': 'tbody td', 'props': [('padding', '8px'), ('border', '1px solid black'), ('color', 'black')]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [('background-color', '#E7E6E6')]},
    {'selector': 'tbody tr:nth-child(even)', 'props': [('background-color', '#F2F2F2')]},
]).set_properties(**{'text-align': 'center'}).hide(axis='index')

display(styled)

Player Name,Points,Correct Scores,Game Week
Murphty,13,3,4
Murphty,13,3,3
